# Lab 06: Landmark Recognition Using SIFT Algorithm

[![Student ID](https://img.shields.io/badge/Student%20ID-FA23--BAI--004-blue)](https://github.com/AtifAliShah)
[![Student](https://img.shields.io/badge/Student-Atif%20Ali%20Shah-darkblue)](https://github.com/AtifAliShah)
[![Repository](https://img.shields.io/badge/Repository-ComputerVision--LABS-teal)](https://github.com/AtifAliShah/ComputerVision-LABS)
[![Course](https://img.shields.io/badge/Course-Computer%20Vision-blueviolet)](https://github.com/AtifAliShah/ComputerVision-LABS)

**Student Name:** Atif Ali Shah  
**Student ID:** FA23-BAI-004  
**Course:** Computer Vision  
**Assignment:** Lab 06 - Landmark Recognition Using SIFT  

---

## 1. Introduction and Objectives

The objective of this laboratory assignment is to develop an automated landmark recognition and geometric localization application using the Scale-Invariant Feature Transform (SIFT) algorithm, Brute-Force feature matching, and RANSAC homography estimation.

### Core Methodology
- **SIFT Feature Detection & Description:** Extraction of scale-space extrema and 128-dimensional gradient orientation histograms.
- **Brute-Force Matching & Lowe's Ratio Test:** k-NN matching ($k=2$) with distance ratio threshold $\tau = 0.75$.
- **Geometric Verification:** Planar homography estimation via RANSAC with geometric sanity checks (convexity, non-degenerate area).
- **Rejection Gate:** Automated rejection of unrecognized and non-database images.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from run_lab06 import task1_load_database, task2_extract_features, task3_process_query, task4_match_and_rank, task5_localize

print(f"OpenCV version: {cv2.__version__}")
sift = cv2.SIFT_create()
db = task1_load_database('data/landmarks')
db = task2_extract_features(db)
print(f"Successfully loaded and indexed {len(db)} landmark categories.")

## 2. Landmark Reference Database Gallery

Multi-view reference database containing 5 famous world landmarks:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (cat, items) in zip(axes, sorted(db.items())):
    rgb = cv2.cvtColor(items[0]['bgr'], cv2.COLOR_BGR2RGB)
    ax.imshow(rgb)
    ax.set_title(cat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. SIFT Keypoints and Scale-Space Extrema

Keypoints detected via Difference-of-Gaussians (DoG) with orientation and scale attributes:

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (cat, items) in zip(axes, sorted(db.items())):
    item = items[0]
    vis = cv2.drawKeypoints(item['bgr'], item['keypoints'][:350], None, color=(0, 255, 0),
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{cat}\n{len(item['keypoints'])} KPs", fontsize=10, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Query Processing, Matching, and Localization

In [ ]:
eiffel_ref = db['eiffel_tower'][0]['bgr']
h, w = eiffel_ref.shape[:2]
M = cv2.getRotationMatrix2D((w/2, h/2), 45, 0.85)
q_rot = cv2.warpAffine(eiffel_ref, M, (w, h), borderMode=cv2.BORDER_REFLECT)

gray_q, kps_q, des_q = task3_process_query(q_rot, sift)
is_rec, top, ranked, msg = task4_match_and_rank(des_q, db)
print(f"Recognition Status: {msg}")

if is_rec and top['best_ref']:
    succ, inliers, ann, lmsg = task5_localize(
        q_rot, kps_q, top['best_ref']['bgr'], top['best_ref']['keypoints'], top['matches']
    )
    print(f"Localization Status: {lmsg} | Inliers: {inliers}")
    plt.figure(figsize=(7, 7))
    plt.imshow(cv2.cvtColor(ann, cv2.COLOR_BGR2RGB))
    plt.title("Localized Landmark Boundary Polygon", fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.show()

## 5. Performance Evaluation Across 6 Environmental Conditions

| Condition | Ground Truth | Predicted Landmark | Query KPs | Matches | Inliers | Decision | Latency |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Original Landmark Image** | `eiffel_tower` | `eiffel_tower` | `2,946` | `2,946` | `2,946` | **RECOGNIZED** | `986.8 ms` |
| **Rotated Landmark Image (45 deg)** | `eiffel_tower` | `eiffel_tower` | `2,517` | `1,234` | `1,227` | **RECOGNIZED** | `801.7 ms` |
| **Resized Landmark Image (0.5x Scale)** | `eiffel_tower` | `eiffel_tower` | `672` | `509` | `508` | **RECOGNIZED** | `236.6 ms` |
| **Reduced Brightness Image (0.35x)** | `eiffel_tower` | `eiffel_tower` | `760` | `718` | `717` | **RECOGNIZED** | `312.0 ms` |
| **Different Viewpoint Image** | `eiffel_tower` | `eiffel_tower` | `1,020` | `1,020` | `1,020` | **RECOGNIZED** | `377.4 ms` |
| **Non-Database Landmark (Big Ben)** | `unknown` | `Unknown / Rejected` | `1,984` | `22` | `0` | **REJECTED** | `666.9 ms` |

### Environmental Robustness Matrix
![Environmental Evaluation](figures/task6_environmental_evaluation.png)

### Latency and Telemetry Summary
![Benchmark Telemetry](figures/task6_benchmark_telemetry.png)

## 6. Analytical Discussion and Limitations

1. **Scale and Rotation Invariance:** The SIFT descriptor successfully mapped rotated queries (1,234 matches, 1,227 inliers) and scaled queries (509 matches, 508 inliers).
2. **Illumination Robustness:** SIFT relies on normalized spatial gradients rather than raw pixel intensities, ensuring stability under 0.35x low-light conditions (718 matches, 717 inliers).
3. **Geometric Rejection Gate:** For non-database landmarks (Big Ben), random background matches failed RANSAC consensus (0 inliers), correctly triggering rejection.
4. **System Limitations:** High computational overhead during exhaustive brute-force search. For industrial-scale databases, Bag-of-Visual-Words or FLANN KD-trees are required.